# 02 Modeling and Inference

## P4 · Baselines and Backtest Harness

This notebook starts from the P3 artifact `data/processed/features.parquet`. It builds the walk-forward folds, baseline forecasts, metric implementations, denominator guards, and the comparison table every later model must beat.

In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

CWD = Path.cwd().resolve()
if (CWD / "data" / "processed" / "features.parquet").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data" / "processed" / "features.parquet").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError("Run this notebook after P3 writes data/processed/features.parquet")

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features.parquet"
P1_SERIES_INVENTORY_PATH = PROJECT_ROOT / "outputs" / "metrics" / "p1_ts_eda" / "series_inventory.csv"
P4_TABLE_DIR = PROJECT_ROOT / "outputs" / "metrics" / "baselines"
P4_TABLE_DIR.mkdir(parents=True, exist_ok=True)

HORIZON_WEEKS = 13
FOLD_COUNT = 4
SEASONAL_PERIOD_WEEKS = 52
MOVING_AVERAGE_WINDOW_WEEKS = 4
DENOMINATOR_FLOOR = 1e-8
QUANTILES = [0.1, 0.5, 0.8, 0.9]
SBA_ALPHA = 0.1
SBA_BIAS_CORRECTION = 1.0 - SBA_ALPHA / 2.0
HIERARCHY_LEVELS = ["total", "state", "city", "store", "cluster", "family", "store_family"]
BASELINE_NAMES = ["naive", "seasonal_naive", "moving_average", "sba"]


### P4 Helper Functions

Metric denominators are computed from each fold's post-truncation training data only. Series with invalid denominators or launches inside a test window are excluded from aggregate scores and counted.

In [2]:
def series_id_for_level(frame: pd.DataFrame, level: str) -> pd.Series:
    if level == "total":
        return pd.Series("total", index=frame.index)
    if level == "state":
        return frame["state"].astype(str)
    if level == "city":
        return frame["city"].astype(str)
    if level == "store":
        return frame["store_nbr"].astype(str)
    if level == "cluster":
        return frame["cluster"].astype(str)
    if level == "family":
        return frame["family"].astype(str)
    if level == "store_family":
        return frame["store_nbr"].astype(str) + "|" + frame["family"].astype(str)
    raise ValueError(f"Unknown level: {level}")


def aggregate_to_level(frame: pd.DataFrame, level: str) -> pd.DataFrame:
    working = frame.copy()
    working["level"] = level
    working["series_id"] = series_id_for_level(working, level)
    grouped = (
        working.groupby(["level", "series_id", "week_start"], observed=True, as_index=False)["sales"]
        .sum()
        .sort_values(["level", "series_id", "week_start"])
        .reset_index(drop=True)
    )
    return grouped


def make_folds(max_week_start: pd.Timestamp, fold_count: int, horizon_weeks: int) -> pd.DataFrame:
    final_test_start = max_week_start - pd.Timedelta(weeks=horizon_weeks - 1)
    rows = []
    for fold_number in range(fold_count):
        test_start = final_test_start - pd.Timedelta(weeks=horizon_weeks * (fold_count - fold_number - 1))
        test_end = test_start + pd.Timedelta(weeks=horizon_weeks - 1)
        rows.append(
            {
                "fold": fold_number + 1,
                "train_end": test_start - pd.Timedelta(weeks=1),
                "test_start": test_start,
                "test_end": test_end,
                "horizon_weeks": horizon_weeks,
            }
        )
    return pd.DataFrame(rows)


def assert_fold_no_lookahead(folds: pd.DataFrame) -> None:
    for fold in folds.itertuples(index=False):
        assert fold.train_end < fold.test_start, f"Fold {fold.fold} train window touches test window"
        assert fold.test_end >= fold.test_start, f"Fold {fold.fold} has invalid test dates"


def training_denominator(values: pd.Series) -> tuple:
    clean = values.astype(float).dropna().to_numpy()
    if len(clean) < 2:
        return np.nan, 0
    differences = np.abs(np.diff(clean))
    return float(np.mean(differences)), len(clean)


def rmsse(actual: np.ndarray, forecast: np.ndarray, denominator: float) -> float:
    return float(np.sqrt(np.mean((actual - forecast) ** 2)) / denominator)


def mase(actual: np.ndarray, forecast: np.ndarray, denominator: float) -> float:
    return float(np.mean(np.abs(actual - forecast)) / denominator)


def pinball_loss(actual: np.ndarray, forecast: np.ndarray, quantile: float) -> float:
    error = actual - forecast
    return float(np.mean(np.maximum(quantile * error, (quantile - 1.0) * error)))


def empirical_coverage(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.mean(actual <= forecast))


def mean_bias(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.mean(forecast - actual))


def naive_forecast(train: pd.Series, horizon: int) -> np.ndarray:
    return np.repeat(float(train.iloc[-1]), horizon)


def seasonal_naive_forecast(full_series: pd.Series, test_index: pd.DatetimeIndex, seasonal_period: int) -> np.ndarray:
    values = []
    for week in test_index:
        source_week = week - pd.Timedelta(weeks=seasonal_period)
        if source_week in full_series.index:
            values.append(float(full_series.loc[source_week]))
        else:
            values.append(float(full_series.loc[full_series.index < week].iloc[-1]))
    return np.asarray(values, dtype=float)


def moving_average_forecast(train: pd.Series, horizon: int, window: int) -> np.ndarray:
    return np.repeat(float(train.tail(window).mean()), horizon)


def sba_forecast(train: pd.Series, horizon: int, alpha: float, correction: float) -> np.ndarray:
    values = train.astype(float).to_numpy()
    non_zero_positions = np.flatnonzero(values > 0)
    if len(non_zero_positions) == 0:
        return np.zeros(horizon, dtype=float)
    first_position = int(non_zero_positions[0])
    demand_estimate = float(values[first_position])
    interval_estimate = float(first_position + 1)
    previous_position = first_position
    for position in non_zero_positions[1:]:
        interval = float(position - previous_position)
        demand_estimate = alpha * float(values[position]) + (1.0 - alpha) * demand_estimate
        interval_estimate = alpha * interval + (1.0 - alpha) * interval_estimate
        previous_position = int(position)
    rate = correction * demand_estimate / interval_estimate
    return np.repeat(max(rate, 0.0), horizon)


def one_step_residuals(train: pd.Series, baseline_name: str) -> np.ndarray:
    values = train.astype(float)
    if baseline_name == "naive":
        forecast = values.shift(1)
    elif baseline_name == "seasonal_naive":
        forecast = values.shift(SEASONAL_PERIOD_WEEKS)
    elif baseline_name == "moving_average":
        forecast = values.shift(1).rolling(MOVING_AVERAGE_WINDOW_WEEKS, min_periods=1).mean()
    elif baseline_name == "sba":
        forecast = pd.Series(np.repeat(float(values.mean()), len(values)), index=values.index)
    else:
        raise ValueError(f"Unknown baseline: {baseline_name}")
    residuals = (values - forecast).dropna().to_numpy(dtype=float)
    if len(residuals) == 0:
        return np.asarray([0.0], dtype=float)
    return residuals


def quantile_forecasts(point_forecast: np.ndarray, residuals: np.ndarray, quantiles: list) -> dict:
    forecasts = {}
    for quantile in quantiles:
        offset = float(np.quantile(residuals, quantile))
        forecasts[quantile] = np.maximum(point_forecast + offset, 0.0)
    return forecasts


def denominator_canary_result() -> pd.DataFrame:
    full = pd.Series([0.0] * 400 + [4.0, 8.0, 5.0, 7.0, 9.0])
    truncated = full.loc[full.ne(0).idxmax():].reset_index(drop=True)
    full_denominator, full_length = training_denominator(full)
    truncated_denominator, truncated_length = training_denominator(truncated)
    assert truncated_denominator > full_denominator
    return pd.DataFrame(
        {
            "case": ["full_with_leading_zeros", "post_truncation_only"],
            "denominator": [full_denominator, truncated_denominator],
            "denominator_length": [full_length, truncated_length],
        }
    )


def metric_unit_test_results() -> pd.DataFrame:
    actual = np.asarray([2.0, 4.0, 6.0])
    forecast = np.asarray([1.0, 5.0, 7.0])
    denominator = 2.0
    rows = [
        {"metric": "rmsse", "observed": rmsse(actual, forecast, denominator), "expected": np.sqrt(1.0) / 2.0},
        {"metric": "mase", "observed": mase(actual, forecast, denominator), "expected": 1.0 / 2.0},
        {"metric": "pinball_q80", "observed": pinball_loss(actual, forecast, 0.8), "expected": float(np.mean([0.8, 0.2, 0.2]))},
        {"metric": "coverage", "observed": empirical_coverage(actual, forecast), "expected": 2.0 / 3.0},
        {"metric": "bias", "observed": mean_bias(actual, forecast), "expected": 1.0 / 3.0},
    ]
    result = pd.DataFrame(rows)
    assert np.allclose(result["observed"], result["expected"])
    return result


### Load Features and Build Folds

The four test windows are non-overlapping 13-week quarters. Every fold trains strictly before its own test start.

In [3]:
features = pd.read_parquet(FEATURE_PATH)
features["week_start"] = pd.to_datetime(features["week_start"])
series_inventory = pd.read_csv(P1_SERIES_INVENTORY_PATH)
segment_lookup = series_inventory.assign(series_id=series_inventory["store_nbr"].astype(str) + "|" + series_inventory["family"].astype(str))[["series_id", "intermittency_segment"]]
assert features["sales"].notna().all()
assert features[["store_nbr", "family", "week_start"]].duplicated().sum() == 0

folds = make_folds(features["week_start"].max(), FOLD_COUNT, HORIZON_WEEKS)
assert_fold_no_lookahead(folds)
folds.to_csv(P4_TABLE_DIR / "fold_boundaries.csv", index=False)
folds

,fold,train_end,test_start,test_end,horizon_weeks
0,1,2016-08-15,2016-08-22,2016-11-14,13
1,2,2016-11-14,2016-11-21,2017-02-13,13
2,3,2017-02-13,2017-02-20,2017-05-15,13
3,4,2017-05-15,2017-05-22,2017-08-14,13


### Metric and Denominator Tests

These are hand-computed checks for the metric implementations and the leading-zero denominator rule.

In [4]:
metric_tests = metric_unit_test_results()
denominator_canary = denominator_canary_result()
metric_tests.to_csv(P4_TABLE_DIR / "metric_unit_tests.csv", index=False)
denominator_canary.to_csv(P4_TABLE_DIR / "denominator_leading_zero_canary.csv", index=False)
metric_tests, denominator_canary

(        metric  observed  expected
 0        rmsse  0.500000  0.500000
 1         mase  0.500000  0.500000
 2  pinball_q80  0.400000  0.400000
 3     coverage  0.666667  0.666667
 4         bias  0.333333  0.333333,
                       case  denominator  denominator_length
 0  full_with_leading_zeros     0.037129                 405
 1     post_truncation_only     2.750000                   5)

### Run Baseline Backtest

Baselines are fit from each fold's training window only. Scores are reported per hierarchy level, per fold, with excluded-series counts.

In [5]:
level_frames = {level: aggregate_to_level(features, level) for level in HIERARCHY_LEVELS}
score_rows = []
forecast_rows = []
excluded_rows = []
launch_rows = []

for level, level_frame in level_frames.items():
    for fold in folds.itertuples(index=False):
        train_end = pd.Timestamp(fold.train_end)
        test_start = pd.Timestamp(fold.test_start)
        test_end = pd.Timestamp(fold.test_end)
        fold_frame = level_frame.loc[level_frame["week_start"] <= test_end]
        for series_id, series_frame in fold_frame.groupby("series_id", observed=True):
            series_frame = series_frame.sort_values("week_start")
            train = series_frame.loc[series_frame["week_start"] <= train_end].set_index("week_start")["sales"]
            test = series_frame.loc[(series_frame["week_start"] >= test_start) & (series_frame["week_start"] <= test_end)].set_index("week_start")["sales"]
            if test.empty:
                continue
            if train.empty:
                launch_rows.append({"level": level, "fold": fold.fold, "series_id": series_id, "reason": "launches_inside_test_window"})
                continue
            denominator, denominator_length = training_denominator(train)
            assert denominator_length == len(train.dropna()), "Denominator length must equal post-truncation training length"
            if not np.isfinite(denominator) or denominator <= DENOMINATOR_FLOOR:
                excluded_rows.append(
                    {
                        "level": level,
                        "fold": fold.fold,
                        "series_id": series_id,
                        "reason": "invalid_denominator",
                        "denominator": denominator,
                        "denominator_length": denominator_length,
                    }
                )
                continue
            if len(test) != HORIZON_WEEKS:
                excluded_rows.append(
                    {
                        "level": level,
                        "fold": fold.fold,
                        "series_id": series_id,
                        "reason": "incomplete_test_horizon",
                        "denominator": denominator,
                        "denominator_length": denominator_length,
                    }
                )
                continue
            full_series = series_frame.set_index("week_start")["sales"]
            test_index = test.index
            actual = test.to_numpy(dtype=float)
            baseline_points = {
                "naive": naive_forecast(train, len(test)),
                "seasonal_naive": seasonal_naive_forecast(full_series, test_index, SEASONAL_PERIOD_WEEKS),
                "moving_average": moving_average_forecast(train, len(test), MOVING_AVERAGE_WINDOW_WEEKS),
                "sba": sba_forecast(train, len(test), SBA_ALPHA, SBA_BIAS_CORRECTION),
            }
            for baseline_name, point_forecast in baseline_points.items():
                residuals = one_step_residuals(train, baseline_name)
                q_forecasts = quantile_forecasts(point_forecast, residuals, QUANTILES)
                row = {
                    "level": level,
                    "fold": fold.fold,
                    "series_id": series_id,
                    "baseline": baseline_name,
                    "rmsse": rmsse(actual, point_forecast, denominator),
                    "mase": mase(actual, point_forecast, denominator),
                    "bias": mean_bias(actual, point_forecast),
                    "denominator": denominator,
                    "denominator_length": denominator_length,
                    "test_observations": len(test),
                }
                for quantile in QUANTILES:
                    row[f"pinball_q{int(quantile * 100)}"] = pinball_loss(actual, q_forecasts[quantile], quantile)
                    row[f"coverage_q{int(quantile * 100)}"] = empirical_coverage(actual, q_forecasts[quantile])
                score_rows.append(row)
                for horizon_step, (week, actual_value, forecast_value) in enumerate(zip(test_index, actual, point_forecast), start=1):
                    forecast_rows.append(
                        {
                            "level": level,
                            "fold": fold.fold,
                            "series_id": series_id,
                            "baseline": baseline_name,
                            "week_start": week,
                            "horizon_step": horizon_step,
                            "actual": actual_value,
                            "point_forecast": forecast_value,
                        }
                    )

baseline_scores_by_series = pd.DataFrame(score_rows)
baseline_forecasts_sample = pd.DataFrame(forecast_rows)
excluded_series_columns = ["level", "fold", "series_id", "reason", "denominator", "denominator_length"]
launch_inside_test_columns = ["level", "fold", "series_id", "reason"]
excluded_series = pd.DataFrame(excluded_rows, columns=excluded_series_columns)
launch_inside_test = pd.DataFrame(launch_rows, columns=launch_inside_test_columns)

baseline_scores_by_series.to_csv(P4_TABLE_DIR / "baseline_scores_by_series.csv", index=False)
baseline_forecasts_sample.to_csv(P4_TABLE_DIR / "baseline_forecasts_point.csv", index=False)
excluded_series.to_csv(P4_TABLE_DIR / "excluded_series.csv", index=False)
launch_inside_test.to_csv(P4_TABLE_DIR / "series_launching_inside_test.csv", index=False)

baseline_scores_by_series.head(), excluded_series.head(), launch_inside_test.head()

(   level  fold series_id        baseline     rmsse      mase           bias  \
 0  total     1     total           naive  1.299769  1.062337 -259009.651059   
 1  total     1     total  seasonal_naive  0.973354  0.751207  -37092.900806   
 2  total     1     total  moving_average  1.300201  1.062514 -259281.294530   
 3  total     1     total             sba  1.517467  1.197457 -379530.674024   
 4  total     2     total           naive  1.985121  1.448242 -458170.343230   
 
      denominator  denominator_length  test_observations    pinball_q10  \
 0  354231.417490                 190                 13   79757.801590   
 1  354231.417490                 190                 13  170940.690251   
 2  354231.417490                 190                 13   81970.090057   
 3  354231.417490                 190                 13  189902.208390   
 4  356288.502919                 203                 13   97296.673191   
 
    coverage_q10    pinball_q50  coverage_q50    pinball_q80  cove

### Baseline Score Tables

This is the bar later models must clear: per level, per fold, with mean and range across folds. Excluded-series counts are included every time.

In [6]:
excluded_counts = (
    excluded_series.groupby(["level", "fold"], observed=True)
    .size()
    .rename("excluded_series_count")
    .reset_index()
    if not excluded_series.empty
    else pd.DataFrame(columns=["level", "fold", "excluded_series_count"])
)
launch_counts = (
    launch_inside_test.groupby(["level", "fold"], observed=True)
    .size()
    .rename("launch_inside_test_count")
    .reset_index()
    if not launch_inside_test.empty
    else pd.DataFrame(columns=["level", "fold", "launch_inside_test_count"])
)

baseline_level_fold = (
    baseline_scores_by_series.groupby(["level", "fold", "baseline"], observed=True)
    .agg(
        rmsse=("rmsse", "mean"),
        mase=("mase", "mean"),
        bias=("bias", "mean"),
        pinball_q10=("pinball_q10", "mean"),
        pinball_q50=("pinball_q50", "mean"),
        pinball_q80=("pinball_q80", "mean"),
        pinball_q90=("pinball_q90", "mean"),
        coverage_q10=("coverage_q10", "mean"),
        coverage_q50=("coverage_q50", "mean"),
        coverage_q80=("coverage_q80", "mean"),
        coverage_q90=("coverage_q90", "mean"),
        scored_series=("series_id", "nunique"),
    )
    .reset_index()
    .merge(excluded_counts, on=["level", "fold"], how="left")
    .merge(launch_counts, on=["level", "fold"], how="left")
)
baseline_level_fold["excluded_series_count"] = baseline_level_fold["excluded_series_count"].fillna(0).astype(int)
baseline_level_fold["launch_inside_test_count"] = baseline_level_fold["launch_inside_test_count"].fillna(0).astype(int)

baseline_summary = (
    baseline_level_fold.groupby(["level", "baseline"], observed=True)
    .agg(
        rmsse_mean=("rmsse", "mean"),
        rmsse_min=("rmsse", "min"),
        rmsse_max=("rmsse", "max"),
        mase_mean=("mase", "mean"),
        mase_min=("mase", "min"),
        mase_max=("mase", "max"),
        bias_mean=("bias", "mean"),
        pinball_q80_mean=("pinball_q80", "mean"),
        coverage_q80_mean=("coverage_q80", "mean"),
        scored_series_min=("scored_series", "min"),
        excluded_series_max=("excluded_series_count", "max"),
        launch_inside_test_max=("launch_inside_test_count", "max"),
    )
    .reset_index()
    .sort_values(["level", "rmsse_mean"])
)

baseline_segment_fold = (
    baseline_scores_by_series.loc[baseline_scores_by_series["level"] == "store_family"]
    .merge(segment_lookup, on="series_id", how="left")
    .groupby(["intermittency_segment", "fold", "baseline"], observed=True)
    .agg(
        rmsse=("rmsse", "mean"),
        mase=("mase", "mean"),
        bias=("bias", "mean"),
        pinball_q80=("pinball_q80", "mean"),
        coverage_q80=("coverage_q80", "mean"),
        scored_series=("series_id", "nunique"),
    )
    .reset_index()
)
baseline_segment_summary = (
    baseline_segment_fold.groupby(["intermittency_segment", "baseline"], observed=True)
    .agg(
        rmsse_mean=("rmsse", "mean"),
        rmsse_min=("rmsse", "min"),
        rmsse_max=("rmsse", "max"),
        mase_mean=("mase", "mean"),
        bias_mean=("bias", "mean"),
        pinball_q80_mean=("pinball_q80", "mean"),
        coverage_q80_mean=("coverage_q80", "mean"),
        scored_series_min=("scored_series", "min"),
    )
    .reset_index()
    .sort_values(["intermittency_segment", "rmsse_mean"])
)
assert baseline_segment_fold["intermittency_segment"].notna().all()

baseline_level_fold.to_csv(P4_TABLE_DIR / "baseline_scores_by_level_fold.csv", index=False)
baseline_summary.to_csv(P4_TABLE_DIR / "baseline_summary_by_level.csv", index=False)
baseline_segment_fold.to_csv(P4_TABLE_DIR / "baseline_scores_by_segment_fold.csv", index=False)
baseline_segment_summary.to_csv(P4_TABLE_DIR / "baseline_summary_by_segment.csv", index=False)

baseline_summary, baseline_segment_summary

(           level        baseline  rmsse_mean  rmsse_min  rmsse_max  mase_mean  \
 0           city  moving_average    2.221838   1.692150   3.104945   1.512522   
 1           city           naive    2.366823   1.692894   3.165717   1.665014   
 3           city  seasonal_naive    2.380411   1.786271   3.495880   1.958920   
 2           city             sba    2.489184   1.749684   3.245227   1.853332   
 5        cluster           naive    2.307011   1.631040   3.323925   1.621174   
 4        cluster  moving_average    2.348952   1.459964   3.301275   1.647976   
 7        cluster  seasonal_naive    2.404707   1.905081   3.185761   1.913935   
 6        cluster             sba    2.555993   1.859727   3.128912   1.932447   
 8         family  moving_average    2.535206   1.562321   3.335748   1.812670   
 10        family             sba    2.692135   1.722682   3.581347   1.950529   
 9         family           naive    2.697144   1.553174   3.373552   1.975001   
 11        famil

### P4 Findings

**Artifacts written**

- Fold boundaries: `outputs/metrics/baselines/fold_boundaries.csv`
- Series-level scores: `outputs/metrics/baselines/baseline_scores_by_series.csv`
- Level/fold comparison table: `outputs/metrics/baselines/baseline_scores_by_level_fold.csv`
- Level summary table: `outputs/metrics/baselines/baseline_summary_by_level.csv`
- Intermittency-segment tables: `baseline_scores_by_segment_fold.csv`, `baseline_summary_by_segment.csv`
- Metric tests and denominator canary: `outputs/metrics/baselines/metric_unit_tests.csv`, `denominator_leading_zero_canary.csv`
- Exclusion reports: `excluded_series.csv`, `series_launching_inside_test.csv`

**P4 exit criteria status**

- Fold boundaries printed and eyeballed against the calendar: done in `fold_boundaries.csv`.
- No fold training window contains test dates: asserted in `assert_fold_no_lookahead`.
- Baseline scores recorded per level and intermittency segment, per fold, with spread: done in the level/fold, segment/fold, and summary tables.
- Metric implementations unit-tested against hand-computed examples: done in `metric_unit_tests.csv`.
- Scaling denominator computed on post-truncation training data only: enforced by reading the P3 post-truncation artifact and checked in `denominator_leading_zero_canary.csv`.
- Denominator floor guard active; excluded-series count reported: done in `excluded_series.csv` and score tables.
- Series launching inside a test window identified and excluded: done in `series_launching_inside_test.csv`.
- The bar every model must clear is now a number on paper: done in `baseline_summary_by_level.csv`.


## P5 · Model Building

This section fits the required model families after P4 has established the comparison bar. Each model emits the configured quantiles, uses the same fold boundaries, records runtime, and compares back to the P4 baselines.

In [7]:
import time

from lightgbm import LGBMRegressor
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX

BASE_FORECAST_DIR = PROJECT_ROOT / "outputs" / "forecasts" / "base"
P5_TABLE_DIR = PROJECT_ROOT / "outputs" / "metrics" / "models"
BASE_FORECAST_DIR.mkdir(parents=True, exist_ok=True)
P5_TABLE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_QUANTILES = QUANTILES
SARIMAX_LEVELS = ["total", "state", "city"]
PROPHET_LEVELS = ["store", "cluster", "family"]
LGBM_LEVEL = "store_family"
SARIMAX_TRAIN_TAIL_WEEKS = 156
PROPHET_TRAIN_TAIL_WEEKS = 156
LGBM_TRAIN_TAIL_WEEKS = 104
LGBM_ESTIMATORS = 120
LGBM_LEARNING_RATE = 0.05
RANDOM_SEED = 20260815


Importing plotly failed. Interactive plots will not work.


### P5 Helper Functions

Model fitting is fold-local. Residual quantiles are estimated from training-window one-step/in-sample residuals only, never from the test window.

In [8]:
def p5_feature_columns() -> list:
    return [
        "onpromotion_sum",
        "promo_flag",
        "holiday_normal",
        "holiday_transferred_original",
        "holiday_transfer_effect",
        "holiday_bridge",
        "holiday_workday",
        "holiday_additional",
        "holiday_event",
        "oil_weekly_mean",
        "oil_weekly_change",
        "week_of_year",
        "month",
        "quarter",
        "year",
    ]


def lgbm_feature_columns() -> list:
    return [
        "sales_lag_1",
        "sales_lag_2",
        "sales_lag_4",
        "sales_lag_13",
        "sales_lag_52",
        "sales_roll_mean_4",
        "sales_roll_mean_8",
        "sales_roll_mean_13",
        "sales_roll_std_4",
        "sales_roll_std_8",
        "sales_roll_std_13",
        "sales_roll_zero_rate_4",
        "sales_roll_zero_rate_8",
        "sales_roll_zero_rate_13",
        "onpromotion_sum",
        "onpromotion_max",
        "promo_flag",
        "promo_lag_1",
        "promo_lag_4",
        "promo_lag_13",
        "promo_roll_mean_4",
        "promo_roll_mean_8",
        "promo_roll_mean_13",
        "holiday_normal",
        "holiday_transferred_original",
        "holiday_transfer_effect",
        "holiday_bridge",
        "holiday_workday",
        "holiday_additional",
        "holiday_event",
        "oil_weekly_mean",
        "oil_weekly_change",
        "oil_change_lag_1",
        "oil_change_lag_4",
        "transactions_lag_1",
        "transactions_lag_4",
        "transactions_lag_13",
        "transactions_roll_mean_4",
        "transactions_roll_mean_13",
        "week_of_year",
        "month",
        "quarter",
        "year",
        "is_year_end_week",
        "store_nbr_code",
        "family_code",
        "city_code",
        "state_code",
        "type_code",
        "cluster_code",
    ]


def aggregate_features_to_level(frame: pd.DataFrame, level: str) -> pd.DataFrame:
    working = frame.copy()
    working["level"] = level
    working["series_id"] = series_id_for_level(working, level)
    aggregations = {"sales": "sum", "onpromotion_sum": "sum", "promo_flag": "max"}
    for column in [
        "holiday_normal",
        "holiday_transferred_original",
        "holiday_transfer_effect",
        "holiday_bridge",
        "holiday_workday",
        "holiday_additional",
        "holiday_event",
    ]:
        aggregations[column] = "max"
    for column in ["oil_weekly_mean", "oil_weekly_change", "week_of_year", "month", "quarter", "year"]:
        aggregations[column] = "first"
    return (
        working.groupby(["level", "series_id", "week_start"], observed=True, as_index=False)
        .agg(aggregations)
        .sort_values(["level", "series_id", "week_start"])
        .reset_index(drop=True)
    )


def metric_rows_for_forecast(
    level: str,
    fold: int,
    series_id: str,
    model_name: str,
    actual: np.ndarray,
    point_forecast: np.ndarray,
    quantile_forecast_map: dict,
    denominator: float,
    runtime_seconds: float,
) -> dict:
    row = {
        "level": level,
        "fold": fold,
        "series_id": series_id,
        "model": model_name,
        "rmsse": rmsse(actual, point_forecast, denominator),
        "mase": mase(actual, point_forecast, denominator),
        "bias": mean_bias(actual, point_forecast),
        "runtime_seconds": runtime_seconds,
        "test_observations": len(actual),
    }
    for quantile in MODEL_QUANTILES:
        forecast_values = quantile_forecast_map[quantile]
        row[f"pinball_q{int(quantile * 100)}"] = pinball_loss(actual, forecast_values, quantile)
        row[f"coverage_q{int(quantile * 100)}"] = empirical_coverage(actual, forecast_values)
    return row


def append_forecast_rows(rows: list, level: str, fold: int, series_id: str, model_name: str, weeks: pd.DatetimeIndex, actual: np.ndarray, point_forecast: np.ndarray, quantile_forecast_map: dict) -> None:
    for position, week in enumerate(weeks):
        row = {
            "level": level,
            "fold": fold,
            "series_id": series_id,
            "model": model_name,
            "week_start": week,
            "horizon_step": position + 1,
            "actual": float(actual[position]),
            "point_forecast": float(point_forecast[position]),
        }
        for quantile in MODEL_QUANTILES:
            row[f"q{int(quantile * 100)}"] = float(quantile_forecast_map[quantile][position])
        rows.append(row)


def fit_sarimax_predict(train_frame: pd.DataFrame, test_frame: pd.DataFrame, exog_columns: list) -> tuple:
    train_tail = train_frame.tail(SARIMAX_TRAIN_TAIL_WEEKS)
    exog_train_all = train_tail[exog_columns].astype(float).ffill().bfill().fillna(0.0)
    variable_exog_columns = exog_train_all.columns[exog_train_all.nunique(dropna=False) > 1].tolist()
    exog_train = exog_train_all[variable_exog_columns] if variable_exog_columns else None
    exog_test = test_frame[variable_exog_columns].astype(float).ffill().bfill().fillna(0.0) if variable_exog_columns else None
    model = SARIMAX(
        train_tail["sales"].astype(float),
        exog=exog_train,
        order=(1, 0, 0),
        seasonal_order=(0, 0, 0, 0),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False, maxiter=30)
    point_forecast = np.maximum(fitted.forecast(steps=len(test_frame), exog=exog_test).to_numpy(dtype=float), 0.0)
    fitted_values = np.asarray(fitted.fittedvalues, dtype=float)
    residuals = train_tail["sales"].to_numpy(dtype=float)[-len(fitted_values):] - fitted_values
    return point_forecast, quantile_forecasts(point_forecast, residuals, MODEL_QUANTILES)


def fit_prophet_predict(train_frame: pd.DataFrame, test_frame: pd.DataFrame, regressor_columns: list) -> tuple:
    train_tail = train_frame.tail(PROPHET_TRAIN_TAIL_WEEKS).copy()
    prophet_train = train_tail.rename(columns={"week_start": "ds", "sales": "y"})[["ds", "y"] + regressor_columns]
    prophet_test = test_frame.rename(columns={"week_start": "ds"})[["ds"] + regressor_columns]
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        uncertainty_samples=0,
        interval_width=0.8,
    )
    for column in regressor_columns:
        model.add_regressor(column)
    model.fit(prophet_train)
    forecast = model.predict(prophet_test)
    point_forecast = np.maximum(forecast["yhat"].to_numpy(dtype=float), 0.0)
    train_pred = model.predict(prophet_train)
    residuals = prophet_train["y"].to_numpy(dtype=float) - train_pred["yhat"].to_numpy(dtype=float)
    return point_forecast, quantile_forecasts(point_forecast, residuals, MODEL_QUANTILES)


def fit_lgbm_quantiles(train_frame: pd.DataFrame, test_frame: pd.DataFrame, feature_columns: list) -> tuple:
    train_tail = train_frame.loc[train_frame["week_start"] >= train_frame["week_start"].max() - pd.Timedelta(weeks=LGBM_TRAIN_TAIL_WEEKS)].copy()
    train_model = train_tail.dropna(subset=feature_columns + ["sales"])
    test_model = test_frame.dropna(subset=feature_columns + ["sales"])
    x_train = train_model[feature_columns].astype(float)
    y_train = train_model["sales"].astype(float)
    x_test = test_model[feature_columns].astype(float)
    predictions = {}
    for quantile in MODEL_QUANTILES:
        model = LGBMRegressor(
            objective="quantile",
            alpha=quantile,
            n_estimators=LGBM_ESTIMATORS,
            learning_rate=LGBM_LEARNING_RATE,
            num_leaves=31,
            min_child_samples=40,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_SEED,
            verbose=-1,
        )
        model.fit(x_train, y_train)
        predictions[quantile] = np.maximum(model.predict(x_test), 0.0)
    point_forecast = predictions[0.5]
    return test_model, point_forecast, predictions


### SARIMAX at Upper Levels

SARIMAX is fitted for total, state, and city series using exogenous promotion, holiday, oil, and calendar columns. Quantiles are residual-based from the training window.

In [9]:
model_metric_rows = []
model_forecast_rows = []
model_issue_rows = []
exog_columns = p5_feature_columns()

for level in SARIMAX_LEVELS:
    level_frame = aggregate_features_to_level(features, level)
    for fold in folds.itertuples(index=False):
        train_end = pd.Timestamp(fold.train_end)
        test_start = pd.Timestamp(fold.test_start)
        test_end = pd.Timestamp(fold.test_end)
        for series_id, series_frame in level_frame.groupby("series_id", observed=True):
            series_frame = series_frame.sort_values("week_start")
            train_frame = series_frame.loc[series_frame["week_start"] <= train_end]
            test_frame = series_frame.loc[(series_frame["week_start"] >= test_start) & (series_frame["week_start"] <= test_end)]
            if len(train_frame) < 52 or len(test_frame) != HORIZON_WEEKS:
                model_issue_rows.append({"model": "sarimax", "level": level, "fold": fold.fold, "series_id": series_id, "reason": "insufficient_history_or_test"})
                continue
            denominator, denominator_length = training_denominator(train_frame["sales"])
            if not np.isfinite(denominator) or denominator <= DENOMINATOR_FLOOR:
                model_issue_rows.append({"model": "sarimax", "level": level, "fold": fold.fold, "series_id": series_id, "reason": "invalid_denominator"})
                continue
            start = time.perf_counter()
            try:
                point_forecast, q_forecast = fit_sarimax_predict(train_frame, test_frame, exog_columns)
            except Exception as exc:
                model_issue_rows.append({"model": "sarimax", "level": level, "fold": fold.fold, "series_id": series_id, "reason": str(exc)[:200]})
                continue
            runtime = time.perf_counter() - start
            actual = test_frame["sales"].to_numpy(dtype=float)
            model_metric_rows.append(metric_rows_for_forecast(level, fold.fold, series_id, "sarimax", actual, point_forecast, q_forecast, denominator, runtime))
            append_forecast_rows(model_forecast_rows, level, fold.fold, series_id, "sarimax", test_frame["week_start"], actual, point_forecast, q_forecast)

pd.DataFrame(model_metric_rows).tail()

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/base/model.py:60

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

...

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_mod

/Users/samirankundu/Desktop/forecasting/.venv/lib/python3.9/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


,level,fold,series_id,model,rmsse,mase,bias,runtime_seconds,test_observations,pinball_q10,coverage_q10,pinball_q50,coverage_q50,pinball_q80,coverage_q80,pinball_q90,coverage_q90
149,city,4,Quevedo,sarimax,2.216935,1.200993,1813.094465,0.075437,13,3278.522274,0.076923,3450.480014,0.384615,2206.203108,0.769231,1320.541721,1.000000
150,city,4,Quito,sarimax,2.443744,1.536279,236531.567622,0.020037,13,122757.527956,0.307692,158249.945286,0.769231,105108.235778,0.846154,66298.888268,1.000000
151,city,4,Riobamba,sarimax,2.592377,1.845120,1082.942350,0.059253,13,2644.404519,0.076923,4040.940982,0.615385,3043.088469,0.615385,1970.941297,0.692308
152,city,4,Salinas,sarimax,1.140763,0.989654,12272.439530,0.036795,13,2500.098137,0.461538,5653.971808,0.846154,4051.357048,1.000000,2766.291474,1.000000
153,city,4,Santo Domingo,sarimax,2.615205,1.803967,11052.218931,0.080286,13,8507.200129,0.307692,11819.254938,0.615385,8742.723368,0.846154,5637.954044,0.846154


### Prophet at Mid Levels

Prophet is fitted for store, cluster, and family series. Holiday structure is included through the P3 holiday regressors, including transferred, bridge, work-day, additional, and event flags.

In [10]:
prophet_regressors = [
    "onpromotion_sum",
    "promo_flag",
    "holiday_normal",
    "holiday_transferred_original",
    "holiday_transfer_effect",
    "holiday_bridge",
    "holiday_workday",
    "holiday_additional",
    "holiday_event",
]

for level in PROPHET_LEVELS:
    level_frame = aggregate_features_to_level(features, level)
    for fold in folds.itertuples(index=False):
        train_end = pd.Timestamp(fold.train_end)
        test_start = pd.Timestamp(fold.test_start)
        test_end = pd.Timestamp(fold.test_end)
        for series_id, series_frame in level_frame.groupby("series_id", observed=True):
            series_frame = series_frame.sort_values("week_start")
            train_frame = series_frame.loc[series_frame["week_start"] <= train_end]
            test_frame = series_frame.loc[(series_frame["week_start"] >= test_start) & (series_frame["week_start"] <= test_end)]
            if len(train_frame) < 52 or len(test_frame) != HORIZON_WEEKS:
                model_issue_rows.append({"model": "prophet", "level": level, "fold": fold.fold, "series_id": series_id, "reason": "insufficient_history_or_test"})
                continue
            denominator, denominator_length = training_denominator(train_frame["sales"])
            if not np.isfinite(denominator) or denominator <= DENOMINATOR_FLOOR:
                model_issue_rows.append({"model": "prophet", "level": level, "fold": fold.fold, "series_id": series_id, "reason": "invalid_denominator"})
                continue
            start = time.perf_counter()
            try:
                point_forecast, q_forecast = fit_prophet_predict(train_frame, test_frame, prophet_regressors)
            except Exception as exc:
                model_issue_rows.append({"model": "prophet", "level": level, "fold": fold.fold, "series_id": series_id, "reason": str(exc)[:200]})
                continue
            runtime = time.perf_counter() - start
            actual = test_frame["sales"].to_numpy(dtype=float)
            model_metric_rows.append(metric_rows_for_forecast(level, fold.fold, series_id, "prophet", actual, point_forecast, q_forecast, denominator, runtime))
            append_forecast_rows(model_forecast_rows, level, fold.fold, series_id, "prophet", test_frame["week_start"], actual, point_forecast, q_forecast)

pd.DataFrame(model_metric_rows).tail()

15:13:26 - cmdstanpy - INFO - Chain [1] start processing


15:13:26 - cmdstanpy - INFO - Chain [1] done processing


15:13:27 - cmdstanpy - INFO - Chain [1] start processing


15:13:27 - cmdstanpy - INFO - Chain [1] done processing


15:13:27 - cmdstanpy - INFO - Chain [1] start processing


...

15:13:50 - cmdstanpy - INFO - Chain [1] start processing


15:13:50 - cmdstanpy - INFO - Chain [1] done processing


15:13:50 - cmdstanpy - INFO - Chain [1] start processing


15:13:50 - cmdstanpy - INFO - Chain [1] done processing


,level,fold,series_id,model,rmsse,mase,bias,runtime_seconds,test_observations,pinball_q10,coverage_q10,pinball_q50,coverage_q50,pinball_q80,coverage_q80,pinball_q90,coverage_q90
556,family,4,POULTRY,prophet,2.865771,1.627036,-1270.696751,0.050140,13,7801.042590,0.076923,8650.370504,0.153846,5120.406984,0.307692,2519.549893,0.769231
557,family,4,PREPARED FOODS,prophet,3.620344,2.026228,4121.580600,0.049418,13,2201.976470,0.538462,2041.914303,0.923077,1123.290564,1.000000,647.329930,1.000000
558,family,4,PRODUCE,prophet,3.077099,2.662301,6178.425125,0.050115,13,59968.177466,0.307692,77457.427344,0.538462,63767.020982,0.615385,50096.026393,0.615385
559,family,4,SCHOOL AND OFFICE SUPPLIES,prophet,11.959237,5.716408,-3241.469113,0.050953,13,368.307946,0.000000,1629.914625,0.153846,2382.528666,0.384615,2617.357652,0.461538
560,family,4,SEAFOOD,prophet,1.851774,1.109341,329.786049,0.052992,13,377.216630,0.230769,443.833158,0.461538,268.289271,0.769231,144.551140,0.769231


### LightGBM Quantile at Bottom Level

LightGBM is pooled across store-family series. Separate quantile models are fitted for each configured quantile on each fold, using only rows before the fold's test start.

In [11]:
lgbm_columns = lgbm_feature_columns()
for fold in folds.itertuples(index=False):
    train_end = pd.Timestamp(fold.train_end)
    test_start = pd.Timestamp(fold.test_start)
    test_end = pd.Timestamp(fold.test_end)
    train_frame = features.loc[features["week_start"] <= train_end].copy()
    test_frame = features.loc[(features["week_start"] >= test_start) & (features["week_start"] <= test_end)].copy()
    assert train_frame["week_start"].max() < test_frame["week_start"].min()
    start = time.perf_counter()
    lgbm_test, point_forecast, q_forecast = fit_lgbm_quantiles(train_frame, test_frame, lgbm_columns)
    runtime = time.perf_counter() - start
    lgbm_test = lgbm_test.copy()
    lgbm_test["series_id"] = lgbm_test["store_nbr"].astype(str) + "|" + lgbm_test["family"].astype(str)
    lgbm_test["point_forecast"] = point_forecast
    for quantile in MODEL_QUANTILES:
        lgbm_test[f"q{int(quantile * 100)}"] = q_forecast[quantile]
    complete_series_ids = lgbm_test.groupby("series_id", observed=True).filter(lambda group: len(group) == HORIZON_WEEKS)["series_id"].nunique()
    runtime_per_series = runtime / max(int(complete_series_ids), 1)
    for series_id, series_frame in lgbm_test.groupby("series_id", observed=True):
        series_frame = series_frame.sort_values("week_start")
        if len(series_frame) != HORIZON_WEEKS:
            model_issue_rows.append({"model": "lightgbm_quantile", "level": LGBM_LEVEL, "fold": fold.fold, "series_id": series_id, "reason": "incomplete_test_horizon_after_missing_features"})
            continue
        train_series = features.loc[(features["week_start"] <= train_end) & ((features["store_nbr"].astype(str) + "|" + features["family"].astype(str)) == series_id), "sales"]
        denominator, denominator_length = training_denominator(train_series)
        if not np.isfinite(denominator) or denominator <= DENOMINATOR_FLOOR:
            model_issue_rows.append({"model": "lightgbm_quantile", "level": LGBM_LEVEL, "fold": fold.fold, "series_id": series_id, "reason": "invalid_denominator"})
            continue
        actual = series_frame["sales"].to_numpy(dtype=float)
        point_values = series_frame["point_forecast"].to_numpy(dtype=float)
        q_map = {quantile: series_frame[f"q{int(quantile * 100)}"].to_numpy(dtype=float) for quantile in MODEL_QUANTILES}
        model_metric_rows.append(metric_rows_for_forecast(LGBM_LEVEL, fold.fold, series_id, "lightgbm_quantile", actual, point_values, q_map, denominator, runtime_per_series))
        append_forecast_rows(model_forecast_rows, LGBM_LEVEL, fold.fold, series_id, "lightgbm_quantile", series_frame["week_start"], actual, point_values, q_map)

pd.DataFrame(model_metric_rows).tail()

,level,fold,series_id,model,rmsse,mase,bias,runtime_seconds,test_observations,pinball_q10,coverage_q10,pinball_q50,coverage_q50,pinball_q80,coverage_q80,pinball_q90,coverage_q90
7019,store_family,4,9|POULTRY,lightgbm_quantile,1.758712,1.058315,116.068837,0.002375,13,166.333663,0.076923,221.420263,0.538462,106.307069,0.692308,72.546425,0.846154
7020,store_family,4,9|PREPARED FOODS,lightgbm_quantile,2.464962,1.504744,32.925255,0.002375,13,38.667658,0.076923,54.797301,0.615385,44.588192,0.769231,20.815134,0.769231
7021,store_family,4,9|PRODUCE,lightgbm_quantile,1.732385,0.778615,466.822151,0.002375,13,782.777556,0.000000,486.200086,0.384615,295.506695,0.769231,176.752843,0.923077
7022,store_family,4,9|SCHOOL AND OFFICE SUPPLIES,lightgbm_quantile,7.843602,3.824437,4.975934,0.002375,13,16.812715,0.076923,63.385695,0.538462,68.668537,0.923077,78.218439,0.923077
7023,store_family,4,9|SEAFOOD,lightgbm_quantile,1.300267,0.853471,6.627311,0.002375,13,7.284218,0.153846,10.502900,0.461538,7.499321,0.923077,15.674806,1.000000


### P5 Results and Baseline Comparison

Each model is compared only at its assigned level. If a model loses to the P4 bar, the loss is recorded as a finding, not tuned away.

In [12]:
model_scores_by_series = pd.DataFrame(model_metric_rows)
model_forecasts_base = pd.DataFrame(model_forecast_rows)
model_issues = pd.DataFrame(model_issue_rows, columns=["model", "level", "fold", "series_id", "reason"])

model_level_fold = (
    model_scores_by_series.groupby(["level", "fold", "model"], observed=True)
    .agg(
        rmsse=("rmsse", "mean"),
        mase=("mase", "mean"),
        bias=("bias", "mean"),
        pinball_q10=("pinball_q10", "mean"),
        pinball_q50=("pinball_q50", "mean"),
        pinball_q80=("pinball_q80", "mean"),
        pinball_q90=("pinball_q90", "mean"),
        coverage_q10=("coverage_q10", "mean"),
        coverage_q50=("coverage_q50", "mean"),
        coverage_q80=("coverage_q80", "mean"),
        coverage_q90=("coverage_q90", "mean"),
        runtime_seconds=("runtime_seconds", "sum"),
        scored_series=("series_id", "nunique"),
    )
    .reset_index()
)
model_summary = (
    model_level_fold.groupby(["level", "model"], observed=True)
    .agg(
        rmsse_mean=("rmsse", "mean"),
        rmsse_min=("rmsse", "min"),
        rmsse_max=("rmsse", "max"),
        mase_mean=("mase", "mean"),
        bias_mean=("bias", "mean"),
        pinball_q80_mean=("pinball_q80", "mean"),
        coverage_q80_mean=("coverage_q80", "mean"),
        runtime_seconds_total=("runtime_seconds", "sum"),
        scored_series_min=("scored_series", "min"),
    )
    .reset_index()
)

baseline_best = (
    baseline_summary.groupby("level", observed=True)["rmsse_mean"]
    .min()
    .rename("best_baseline_rmsse_mean")
    .reset_index()
)
model_vs_baseline = model_summary.merge(baseline_best, on="level", how="left")
model_vs_baseline["beats_best_baseline"] = model_vs_baseline["rmsse_mean"] < model_vs_baseline["best_baseline_rmsse_mean"]
model_vs_baseline["finding"] = np.where(
    model_vs_baseline["beats_best_baseline"],
    "beats P4 baseline bar",
    "does not beat P4 baseline bar; recorded as finding, not tuned away",
)

assert set(MODEL_QUANTILES) == {0.1, 0.5, 0.8, 0.9}
for quantile in MODEL_QUANTILES:
    assert f"q{int(quantile * 100)}" in model_forecasts_base.columns
assert not model_scores_by_series.empty
assert model_level_fold["fold"].nunique() == FOLD_COUNT

model_scores_by_series.to_csv(P5_TABLE_DIR / "model_scores_by_series.csv", index=False)
model_level_fold.to_csv(P5_TABLE_DIR / "model_scores_by_level_fold.csv", index=False)
model_summary.to_csv(P5_TABLE_DIR / "model_summary_by_level.csv", index=False)
model_vs_baseline.to_csv(P5_TABLE_DIR / "model_vs_baseline.csv", index=False)
model_issues.to_csv(P5_TABLE_DIR / "model_issues.csv", index=False)
model_forecasts_base.to_parquet(BASE_FORECAST_DIR / "base_model_forecasts.parquet", index=False)
model_forecasts_base.head(), model_vs_baseline

(   level  fold series_id    model week_start  horizon_step        actual  \
 0  total     1     total  sarimax 2016-08-22             1  4.901269e+06   
 1  total     1     total  sarimax 2016-08-29             2  5.750789e+06   
 2  total     1     total  sarimax 2016-09-05             3  5.492016e+06   
 3  total     1     total  sarimax 2016-09-12             4  5.084586e+06   
 4  total     1     total  sarimax 2016-09-19             5  5.010213e+06   
 
    point_forecast           q10           q50           q80           q90  
 0    5.173005e+06  4.678049e+06  5.082085e+06  5.415505e+06  5.906378e+06  
 1    5.368942e+06  4.873986e+06  5.278022e+06  5.611442e+06  6.102315e+06  
 2    5.647952e+06  5.152996e+06  5.557031e+06  5.890452e+06  6.381325e+06  
 3    5.910117e+06  5.415161e+06  5.819197e+06  6.152618e+06  6.643490e+06  
 4    6.350440e+06  5.855484e+06  6.259519e+06  6.592940e+06  7.083813e+06  ,
           level              model  rmsse_mean  rmsse_min  rmsse_max  \


### P5 Findings

**Artifacts written**

- Base model forecasts: `outputs/forecasts/base/base_model_forecasts.parquet`
- Model score tables: `outputs/metrics/models/model_scores_by_series.csv`, `model_scores_by_level_fold.csv`, `model_summary_by_level.csv`
- Baseline comparison: `outputs/metrics/models/model_vs_baseline.csv`
- Runtime and issue records: `runtime_seconds` columns plus `model_issues.csv`

**P5 exit criteria status**

- Every model beats its baseline at its own level, or the loss is recorded: done in `model_vs_baseline.csv`.
- Every model outputs the configured quantile set: asserted for q10, q50, q80, q90.
- Quantile calibration checked: done via coverage columns, especially `coverage_q80`.
- Per-level, per-fold results with spread: done in `model_scores_by_level_fold.csv` and `model_summary_by_level.csv`.
- Runtime per fold recorded: done in `runtime_seconds`.
- No-lookahead assertion passes inside every fit: folds use train dates strictly before test dates, asserted before LightGBM and inherited from P4 folds for SARIMAX/Prophet.


## P6 · Reconciliation

Not started. P6 begins only after P5 exit criteria are accepted.

## P7 · Evaluation and Selection

Not started. P7 begins only after P6 exit criteria are accepted.